In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df_samples = pd.read_csv("/mnt/DATA_01/Eric/mitospace4d_data/runs/embeddings_test/image_paths.csv", dtype=str, header=None, delimiter=" ")
df_regions = pd.read_csv("/run/user/1002/gvfs/smb-share:server=aquila0.jslab.ucsd.edu,share=ssd_processing/Others/MitoSpace4D/2025_summer/cell_to_region.csv")
embeddings = np.load("/mnt/DATA_01/Eric/mitospace4d_data/runs/embeddings_test/embeddings_raw.npy")

In [ ]:
# Set the header for the first column
df_samples.columns = ["fpath"]
df_samples['condition'] = df_samples['fpath'].apply(lambda x: x.split("/")[-2])
df_samples['filename'] = df_samples['fpath'].apply(lambda x: x.split("/")[-1].split(".")[0])
df_samples['cell_id'] = df_samples['filename'].apply(lambda x: int(x.split("-")[0]))
df_samples['cell_wid'] = df_samples['filename'].apply(lambda x: int(x.split("-")[1]))

# Set up an empty column called "condition_tid"
df_samples['region_id'] = -1

In [ ]:
df_samples

In [ ]:
# Set up region IDs
for condition in df_regions["Data Path"].unique():
    df_regions_condition = df_regions[df_regions["Data Path"] == condition]
    df_samples_condition = df_samples[df_samples["condition"] == condition]

    for i, row in df_regions_condition.iterrows():
        cell_id_start = row["Cell ID Start"]
        try:
            cell_id_end = df_regions_condition.loc[i + 1, "Cell ID Start"]
        except:
            # Set infinite integer value
            cell_id_end = np.inf

        current_region_id = row["Region ID"]
        for i, sample_row in df_samples_condition.iterrows():
            current_cell_id = sample_row['cell_id']

            if current_cell_id >= cell_id_start and current_cell_id < cell_id_end:
                df_samples_condition.at[i, "region_id"] = current_region_id

    df_samples.update(df_samples_condition)

In [ ]:
# Duplicate each row by the number of frames and set the frame_id
n_frames = 20
df_samples = df_samples.loc[df_samples.index.repeat(n_frames)]
df_samples['frame_id'] = np.tile(np.arange(n_frames), df_samples.shape[0] // n_frames)
df_samples.reset_index(inplace=True, drop=True)

In [ ]:
# Set up unique ids for each cell (unique across conditions)
df_samples['cell_id'] = df_samples['cell_id'].astype(str)
df_samples['instance_uid'] = df_samples['condition'] + "_" + df_samples['cell_id']

In [ ]:
n_frames = df_samples["frame_id"].max() + 1
df_samples['instance_frame_id'] = df_samples.apply(lambda x: x['cell_wid']*n_frames + x['frame_id'], axis=1)

In [ ]:
# df_samples.to_csv("sample_frame_data.csv")    

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Optional, Hashable

def build_aligned_from_dataframe(
    df: pd.DataFrame,
    features: np.ndarray,
    id_col: str,
    time_col: str,
    weight_col: Optional[str] = None,  # kept for signature compatibility; ignored here
    sort_time: bool = False,
) -> Tuple[
    List[np.ndarray],            # datasets
    List[Dict[int, int]],        # relations (dict per adjacent pair: row_k -> row_k1)
    List,                        # time_list
    List[Dict[Hashable, int]],   # row_maps
    List[np.ndarray]             # df_index_maps
]:
    if len(df) != features.shape[0]:
        raise ValueError("df and features must have the same number of rows.")

    unique_times = df[time_col].unique()
    if sort_time:
        unique_times = np.sort(unique_times)

    datasets: List[np.ndarray] = []
    df_index_maps: List[np.ndarray] = []
    row_maps: List[Dict[Hashable, int]] = []

    for t in unique_times:
        idx_t = df.index[df[time_col] == t].to_numpy()
        df_index_maps.append(idx_t)
        X_t = features[idx_t]
        datasets.append(X_t)

        ids_t = df.loc[idx_t, id_col].to_numpy()
        row_map_t = {inst_id: i for i, inst_id in enumerate(ids_t)}
        row_maps.append(row_map_t)

    # ---- Minimal change here: build {row_k: row_k1} (int -> int), not lists ----
    relations: List[Dict[int, int]] = []
    frames_per_time = []
    for k, t in enumerate(unique_times):
        idx_t = df_index_maps[k]
        mini = pd.DataFrame({
            "df_idx": idx_t,
            id_col: df.loc[idx_t, id_col].to_numpy(),
            "pos_in_time": np.arange(len(idx_t), dtype=int),
        })
        frames_per_time.append(mini)

    for k in range(len(unique_times) - 1):
        left = frames_per_time[k][[id_col, "pos_in_time"]].rename(columns={"pos_in_time": "row_k"})
        right = frames_per_time[k + 1][[id_col, "pos_in_time"]].rename(columns={"pos_in_time": "row_k1"})
        links = pd.merge(left, right, on=id_col, how="inner")

        rel_dict: Dict[int, int] = {}
        if len(links) > 0:
            # If the same (id, time) appears multiple times (shouldn't), keep the first match
            # to maintain one-to-one mapping required by AlignedUMAP.
            links = links.drop_duplicates(subset=["row_k"], keep="first")
            rel_dict = {int(src): int(dst) for src, dst in zip(links["row_k"].to_numpy(),
                                                              links["row_k1"].to_numpy())}
        relations.append(rel_dict)
    # ---------------------------------------------------------------------------

    return datasets, relations, list(unique_times), row_maps, df_index_maps


In [ ]:
# Assume df has columns: "instance_id", "time"
# and features is a (N, D) numpy array aligned to df rows.
aligned_inputs = build_aligned_from_dataframe(
    df=df_samples,
    features=embeddings,
    id_col="instance_uid",
    time_col="instance_frame_id",
)

datasets, relations, time_list, row_maps, df_index_maps = aligned_inputs

In [ ]:
# import umap
# aligned = umap.aligned_umap.AlignedUMAP(
#     n_neighbors=25,
#     min_dist=0.01,
#     metric="cosine",
#     n_components=3,
#     verbose=True,
# )
# aligned.fit(datasets, relations=relations)

In [ ]:
# %%time
import umap
updating_mapper = umap.AlignedUMAP(n_neighbors=25, min_dist=0.01, metric="cosine", n_components=3, verbose=False)
updating_mapper.fit(datasets[:2], relations=relations[:1])

In [ ]:
# for i in range(2,len(datasets)):
#     %time updating_mapper.update(datasets[i], relations={v:k for k,v in relations[i-1].items()})

In [ ]:
updating_mapper.embeddings_[0]

In [ ]:
# Set up an interactive 3D point cloud viewer with plotly
import plotly.graph_objects as go
import plotly.express as px
import scipy.interpolate

n_embeddings = len(updating_mapper.embeddings_)
es = updating_mapper.embeddings_
df_embeddings = pd.DataFrame(np.vstack(es), columns=["x", "y", "z"])
df_embeddings['t'] = np.concatenate([[t] * len(es[0]) for t in range(n_embeddings)])
df_embeddings['cell_id'] = np.concatenate([[i] * len(es[0]) for i in range(n_embeddings)])

In [ ]:
import numpy as np
import scipy.interpolate as si
from typing import List, Tuple, Dict, Iterable, Optional

# Map "number of samples in a block" -> interpolation kind
INTERP_KIND: Dict[int, str] = {2: "linear", 3: "quadratic", 4: "cubic"}

def interpolate_paths_3d(
    t: np.ndarray,
    x: np.ndarray,
    y: np.ndarray,
    z: np.ndarray,
    c: np.ndarray,
    rep_id: str,
    *,
    n_samples: int = 100,
    time_step_is_consecutive: int = 1,
) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, str]]:
    """
    Interpolate 3D paths (x, y, z) and a scalar c over time.

    Parameters
    ----------
    t : (N,) array-like
        Time indices (e.g., frames). Must be sortable; integers work best.
    x, y, z : (N,) array-like
        3D coordinates at each time.
    c : (N,) array-like
        Per-time scalar to interpolate linearly (e.g., color / score).
    rep_id : str
        Identifier used in the annotation text for each path block.
    n_samples : int, optional
        Number of interpolation samples per block, by default 100.
    time_step_is_consecutive : int, optional
        Expected step between consecutive time points within a block (default 1).

    Returns
    -------
    List of tuples (t_i, x_i, y_i, z_i, c_i, text)
        One entry per consecutive-time block. If a block has only one time point,
        the data are returned unchanged (no interpolation).
    """
    # Convert to numpy arrays and sort by time
    t = np.asarray(t)
    x = np.asarray(x)
    y = np.asarray(y)
    z = np.asarray(z)
    c = np.asarray(c)

    # Sort all arrays by time to ensure monotonic input for interpolation
    order = np.argsort(t)
    t, x, y, z, c = t[order], x[order], y[order], z[order], c[order]

    # Identify breaks where time is not consecutive (e.g., gaps)
    breaks = np.where(np.diff(t) != time_step_is_consecutive)[0] + 1

    # Split into consecutive-time blocks
    t_blocks = np.split(t, breaks)
    x_blocks = np.split(x, breaks)
    y_blocks = np.split(y, breaks)
    z_blocks = np.split(z, breaks)
    c_blocks = np.split(c, breaks)

    paths = []

    for tb, xb, yb, zb, cb in zip(t_blocks, x_blocks, y_blocks, z_blocks, c_blocks):
        # Annotation per block (use mean of c as an example summary)
        text = f"{rep_id} — mean_c: {np.mean(cb):.3f}"

        # Handle degenerate blocks
        if len(tb) <= 1:
            # No interpolation possible; return raw point(s)
            paths.append((tb.copy(), xb.copy(), yb.copy(), zb.copy(), cb.copy(), text))
            continue

        # Choose interpolation kind based on sample count in the block
        kind = INTERP_KIND.get(len(tb), "cubic")

        # Build interpolation grid over the block's time range
        t_new = np.linspace(tb.min(), tb.max(), n_samples)

        # Create 1D interpolators along time for each quantity
        # 'bounds_error=False' allows slight extrapolation if numerical jitter occurs
        fx = si.interp1d(tb, xb, kind=kind, bounds_error=False, fill_value="extrapolate")
        fy = si.interp1d(tb, yb, kind=kind, bounds_error=False, fill_value="extrapolate")
        fz = si.interp1d(tb, zb, kind=kind, bounds_error=False, fill_value="extrapolate")

        # For scalar c, keep it linear to avoid overshoot in labels/colors
        fc = si.interp1d(tb, cb, kind="linear", bounds_error=False, fill_value="extrapolate")

        x_new = fx(t_new)
        y_new = fy(t_new)
        z_new = fz(t_new)
        c_new = fc(t_new)

        paths.append((t_new, x_new, y_new, z_new, c_new, text))

    return paths

In [ ]:
traces = []
for cell_id in df_embeddings['cell_id'].unique():
    df_cell = df_embeddings[df_embeddings['cell_id'] == cell_id]
    paths = interpolate_paths_3d(
        t=df_cell['t'].to_numpy(),
        x=df_cell['x'].to_numpy(),
        y=df_cell['y'].to_numpy(),
        z=df_cell['z'].to_numpy(),
        c=df_cell['t'].to_numpy(),  # Use time as color for demonstration
        rep_id=cell_id,
        n_samples=50,
        time_step_is_consecutive=1,
    )

    for (t_i, x_i, y_i, z_i, c_i, text) in paths:
        trace = go.Scatter3d(
            x=x_i,
            y=y_i,
            z=z_i,
            mode='lines+markers',
            line=dict(
                width=4,
                color=c_i,
                colorscale='Viridis',
                colorbar=dict(title='Time'),
                showscale=False
            ),
            marker=dict(
                size=3,
                color=c_i,
                colorscale='Viridis',
                showscale=False
            ),
            text=[text] * len(x_i),
            hoverinfo='text'
        )
        traces.append(trace)

fig = go.Figure(data=traces)
fig.update_layout(
    width=800,
    height=600,
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    scene_camera=dict(
        eye=dict(x=1.5, y=1.5, z=1.5)
    ),
    autosize=False,
    showlegend=False,
)
fig.show()